# RMY Generation on Google Colab

This notebook generates a Representative Meteorological Year (RMY) file with embedded extreme events (heat waves and cold spells) using the `rmy` package, then plots every detected event.

By default it runs on the Boston sample data in the repository:

- `data/EPWs/` → AMY EPW files, one per year (the year must appear in each file name)
- `data/base/` → exactly one base TMY EPW file

To use your own files, upload them to Colab and change `EPW_DIR` and `BASE_DIR` in section 2.0.

# **0.0 Dependencies**

In [ ]:
!pip install -q pandas numpy scipy matplotlib pytest

# **1.0 Clone the repository**

`BRANCH` selects which version of the code to run. Use `main` for the released version, or a feature branch to check work in progress.

In [ ]:
BRANCH = "main"

%cd /content
!rm -rf RMY
!git clone -b {BRANCH} https://github.com/Nadatarkhan/RMY.git
%cd /content/RMY
!git log -1 --format='Running commit %h: %s'

Optional check: run the package tests. They should all pass before you trust the output.

In [ ]:
!python -m pytest -q tests/

# **2.0 Run the RMY method**

In [ ]:
from rmy import run_full_rmy_pipeline

EPW_DIR = "/content/RMY/data/EPWs"    # AMY EPWs
BASE_DIR = "/content/RMY/data/base"   # one base TMY EPW
OUTPUT_DIR = "/content/output"        # hotspells/, coldspells/ and the RMY EPW

rmy_path = run_full_rmy_pipeline(EPW_DIR, BASE_DIR, OUTPUT_DIR)

# **3.0 Visualization: Extreme Event Explorer**

One circle per detected event, at its start date. Circle size is the event duration. Heat waves are colored by average heat index (°C); cold spells by average wind speed.

Use `heatwave_events.csv` / `coldspells_events.csv` for every AMY year, or the `_peak.csv` files for the events inserted into the RMY.

In [ ]:
from rmy.visualization import visualize_extreme_events

visualize_extreme_events(
    f"{OUTPUT_DIR}/hotspells/heatwave_events.csv",
    f"{OUTPUT_DIR}/coldspells/coldspells_events.csv",
    output_path=f"{OUTPUT_DIR}/event_timeline.png",
)

# **4.0 Download the results**

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/rmy_output", "zip", OUTPUT_DIR)
files.download(archive)